In [42]:
import torch
from master_model import MasterModel
from ttokenizer import Tokenizer

u_tokenizer = Tokenizer("tokenizer.json")
prompt = "the capital of united"
tokens = u_tokenizer.encode(prompt)
tokens.shape, tokens


(torch.Size([7]), tensor([ 0, 61,  1, 61,  2, 61,  3]))

In [31]:
torch.manual_seed(1)
u_model = MasterModel(vocab_size=len(u_tokenizer.vocab), embedding_dim=4, num_heads=2, context_length=32)

sentence_meanings_with_attention_context = u_model(tokens)
sentence_meanings_with_attention_context

tensor([[-0.1530, -0.3205,  0.1112,  0.1276],
        [-0.4967, -0.2912, -0.0172,  0.1406],
        [-0.1390, -0.3184,  0.1232,  0.1325],
        [-0.2729, -0.3094,  0.0994,  0.1647],
        [-0.4454, -0.2996,  0.0044,  0.1424],
        [-0.5217, -0.2850, -0.0500,  0.1161],
        [-0.3271, -0.3152, -0.0267,  0.0628]], grad_fn=<AddmmBackward0>)

In [32]:
sentence_meanings_with_attention_context.shape

torch.Size([7, 4])

In [7]:
u_model

MasterModel(
  (embedding): Embedding(64, 4)
  (pos_embedding): Embedding(32, 4)
  (self_attention): MultiHeadAttention(
    (multi_head_attention): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=4, out_features=4, bias=True)
    )
    (projection): Linear(in_features=4, out_features=4, bias=True)
  )
  (norm): LayerNorm()
  (mlp): MLP(
    (gate_proj): Linear(in_features=4, out_features=4, bias=True)
    (up_proj): Linear(in_features=4, out_features=4, bias=True)
    (down_proj): Linear(in_features=4, out_features=4, bias=True)
    (gelu): GELU()
  )
)

In [33]:
u_model.embedding(tokens)

tensor([[-1.5256e+00, -7.5023e-01, -6.5398e-01, -1.6095e+00],
        [-7.4484e-01, -2.0216e-01, -2.2967e-01,  1.3313e-03],
        [-1.0017e-01, -6.0919e-01, -9.7977e-01, -1.6091e+00],
        [-7.4484e-01, -2.0216e-01, -2.2967e-01,  1.3313e-03],
        [-7.1214e-01,  3.0372e-01, -7.7731e-01, -2.5145e-01],
        [-7.4484e-01, -2.0216e-01, -2.2967e-01,  1.3313e-03],
        [-2.2227e-01,  1.6871e+00,  2.2843e-01,  4.6764e-01]],
       grad_fn=<EmbeddingBackward0>)

In [34]:
tokens.unsqueeze(0)

tensor([[ 0, 61,  1, 61,  2, 61,  3]])

In [35]:
torch.manual_seed(1)

out_tokens = u_model(tokens)

In [36]:
out_tokens

tensor([[-0.1370, -0.3226,  0.1265,  0.1362],
        [-0.4354, -0.2978, -0.0487,  0.0829],
        [-0.2134, -0.3161,  0.1186,  0.1598],
        [-0.2499, -0.2604,  0.1531,  0.1925],
        [-0.3325, -0.2820,  0.0776,  0.1607],
        [-0.4992, -0.2880, -0.0536,  0.1033],
        [-0.5145, -0.2847, -0.0511,  0.1116]], grad_fn=<AddmmBackward0>)

In [12]:
#multihead, birden fazla baglamda inceleme. bu olmaz ise tek bi cumle icindeki baglama bakiyoruz.

In [13]:

import torch.nn as nn
from casual_self_attention import CasualSelfAttention

class DenemeMultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim, output_dim, context_length, num_heads, dropout_rate=0.5):
        super().__init__()
        assert embedding_dim % num_heads == 0

        head_dim = embedding_dim // num_heads
        self.heads = nn.ModuleList(
            [CasualSelfAttention(
                embedding_dim, head_dim, context_length, dropout_rate) for _ in range(num_heads)]
        )
        self.projection = nn.Linear(embedding_dim, output_dim)

    def forward(self, x):
        attention_outs = []
        for head in self.heads:
            head_out = head(x)
            attention_outs.append(head_out)
        concated_attention_outs = torch.concat(attention_outs, dim=1)
        return self.projection(concated_attention_outs)

deneme_multi_head_attention = DenemeMultiHeadAttention(4,4,32,2,dropout_rate=0)


In [14]:
out = deneme_multi_head_attention(torch.randn(4,4))
out.shape, out

(torch.Size([4, 4]),
 tensor([[ 0.0328,  0.2228, -0.4448,  0.4079],
         [ 0.7880,  0.8728, -0.2252,  0.2310],
         [ 0.2310,  0.5523, -0.3674,  0.3774],
         [ 0.2956,  0.6542, -0.3337,  0.3700]], grad_fn=<AddmmBackward0>))

In [15]:
test = DenemeMultiHeadAttention(4,4,32,4,dropout_rate=0)
test_out = test(torch.randn(4,4))
test_out.shape, test_out

(torch.Size([4, 4]),
 tensor([[-0.0893, -0.0330, -0.2677, -0.6383],
         [-0.1622,  0.0673, -0.3046, -0.8201],
         [-0.1491,  0.0586, -0.2811, -0.5314],
         [-0.1700,  0.0228, -0.2607, -0.1932]], grad_fn=<AddmmBackward0>))

In [16]:
from norm_layer import LayerNorm
norm_layer = LayerNorm(4)
#norm_layer(out_tokens)
norm_layer(sentence_meanings_with_attention_context)

tensor([[-0.5022, -1.3944,  0.9048,  0.9918],
        [-1.3463, -0.5091,  0.6065,  1.2489],
        [-0.4679, -1.4158,  0.9172,  0.9665],
        [-0.9065, -1.0778,  0.8391,  1.1452],
        [-1.2644, -0.6412,  0.6581,  1.2475],
        [-1.3965, -0.4145,  0.5608,  1.2501],
        [-1.0171, -0.9481,  0.7236,  1.2416]], grad_fn=<MulBackward0>)

In [17]:
nn.Parameter(torch.ones(4,3)) #matirsimiz

Parameter containing:
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]], requires_grad=True)

In [18]:
example_tensor = torch.tensor([[1,2,3], [4,5,6]], dtype=torch.float32)

mean = example_tensor.mean(dim=-1, keepdim=True)
var = example_tensor.var(dim=-1, keepdim=True, unbiased=False)

mean, var, example_tensor 


(tensor([[2.],
         [5.]]),
 tensor([[0.6667],
         [0.6667]]),
 tensor([[1., 2., 3.],
         [4., 5., 6.]]))

In [19]:
from transformers import AutoTokenizer, AutoModelForCausalLM

q_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
q_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")


/Users/gonul/llm-from-scratch/.llm_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 6894.92it/s]


In [20]:
q_model #Qwen3DecoderLayer 28 defa tekrar ediyor. 28 layer var. 28 defa blok ile for dongusu var. her biri bi konu. 
#"domatesler olgunlasti" cumlesinde hangi ay bilgisi yok, bu blokta bunu cikaracak bilgi olmasi gerek.
# bazi bilgiler cumlenin icinde degil, bu blokta cikarilacaktir. baglama gore temalar cikar.
# ziraatodasinda,ciftciler, tarim bakanliginda, tarim universitesinde bu bloklarin her biri farkli bir bağlama bakiyor.
# egitim verisinde gokyuzundeki ay mi? bahar mi? yoksa tarim mi? gibi farkli baglamlar var. bu bloklarin her biri farkli baglamlara bakiyor.
# 28 baglam, her katmada bi bilgi tutacak. blok olusturmak gerek.

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [21]:
#mlp katmanini yapiyoruz multilayerperceptron, feed forward 
#weightleri egiticez, activation func vs ayarlicaz


![transformers-arch](https://deeprevision.github.io/posts/001-transformer/transformer.png)

![GELU-func](https://miro.medium.com/v2/resize:fit:1400/format:webp/1*O5E-huBuY1UTHMmM--rhLQ.png)

In [22]:
import torch
import torch.nn as nn
class GELU(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x):
        return 0.5*x*(1+torch.tanh(
            torch.sqrt(torch.tensor(2/torch.pi))*(x+0.044715*torch.pow(x,3))
            )
        )
gelu = GELU()
example_tensor = torch.tensor([[1,2,3],[4,5,6]], dtype=torch.float32)
gelu(example_tensor)


tensor([[0.8412, 1.9546, 2.9964],
        [3.9999, 5.0000, 6.0000]])

In [23]:
import torch.nn.functional as F
F.gelu(example_tensor, approximate="tanh")

tensor([[0.8412, 1.9546, 2.9964],
        [3.9999, 5.0000, 6.0000]])

In [24]:
#feed neuron network
class MLP(nn.Module):
    def __init__(self, embedding_dim, hidden_dim):
        super().__init__()
        self.gate_proj = nn.Linear(embedding_dim, hidden_dim)
        self.up_proj = nn.Linear(embedding_dim, hidden_dim)
        self.down_proj = nn.Linear(hidden_dim, embedding_dim)
        #self.layers = nn.Sequential(
        #    nn.Linear(embedding_dim, hidden_dim),
        #    GELU(),
        #    nn.Linear(hidden_dim, embedding_dim)
        #)
        self.gelu = GELU()
    
    def forward(self, x):
        gate = self.gate_proj(x)
        gate = self.gelu(gate)
        up = self.up_proj(x)
        fuse = gate * up
        outputs = self.down_proj(fuse)
        return outputs

In [ ]:
#28 tane baglam olusturmak gerek. 
# transformer bloklari, multihead attention, mlp, layernorm, residual connection ile birlestirip transformer layer olusturmak gerek.
# Decoder blok olusturulacak. 28 tane baglam olacak. her baglam farkli bir konuya bakacak.
import torch
import torch.nn as nn
from multi_head_attention import MultiHeadAttention   
class DecoderBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, context_length, dropout_rate=0.5):
        super().__init__()
        self.self_attention = MultiHeadAttention(embedding_dim, embedding_dim, context_length, num_heads, dropout_rate=0.5)
        self.norm1 = LayerNorm(embedding_dim)
        self.mlp = MLP(embedding_dim, embedding_dim)
        self.norm2 = LayerNorm(embedding_dim)
    
    def forward(self, x):
        res = x # ilk inputi tutuyoruz, residual connection icin.
        res_norm = self.norm1(res)
        
        x = self.self_attention(x)
        x = self.norm1(x)
        
        x = x + res_norm # residual connection ile birlestiriyoruz.
        
        res = self.norm2(x)
        x = self.mlp(x)
        x = self.norm2(x)
        
        x = x + res # residual connection ile birlestiriyoruz.
        
        return x
        
example_tensor = torch.tensor([[1,2,3,4],[5,6,7,8]], dtype=torch.float32)
decoder_block = DecoderBlock(embedding_dim=4, num_heads=2, context_length=3)
decoder_block(example_tensor.unsqueeze(0))
#decoder_block(tokens)

tensor([[[-1.2668,  1.2805,  0.2301, -0.2438],
         [-2.3135,  1.8062,  1.4959, -0.9886]]], grad_fn=<AddBackward0>)